In [18]:
!pip install -q transformers librosa wandb
!git clone https://github.com/Rudrabha/Wav2Lip.git 2>/dev/null || true
!mkdir -p Wav2Lip/checkpoints

!wget -q -nc "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip.pth" -O Wav2Lip/checkpoints/wav2lip.pth
!wget -q -nc "https://huggingface.co/Nekochu/Wav2Lip/resolve/main/lipsync_expert.pth" -O Wav2Lip/checkpoints/lipsync_expert.pth
!ls -lh Wav2Lip/checkpoints/*.pth

-rw-r--r-- 1 root root 189M Mar  8 07:00 Wav2Lip/checkpoints/lipsync_expert.pth
-rw-r--r-- 1 root root 416M Mar  8 07:00 Wav2Lip/checkpoints/wav2lip.pth


In [19]:
import sys
sys.path.insert(0, "/content")
sys.path.insert(0, "/content/Wav2Lip")

import gc
import json
import warnings
from pathlib import Path

import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import wandb
from torch.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from emotion_utils import (
    DifferentiableVideoPreprocess,
    EmotionAgreementMetric,
    load_frozen_audio_encoder,
    load_frozen_video_encoder,
    extract_audio_embedding,
    extract_video_embedding,
)
from models.wav2lip import Wav2Lip as Wav2LipModel


class CrossModalEmotionLoss(nn.Module):
    """F.normalize removed — redundant before F.cosine_similarity."""
    def __init__(self, weight=1.0):
        super().__init__()
        self.weight = weight

    def forward(self, audio_emb, video_emb):
        return self.weight * (1.0 - F.cosine_similarity(audio_emb, video_emb, dim=-1)).mean()

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
METADATA = "/content/processed_data/metadata.json"
WAV2LIP_CKPT = "/content/Wav2Lip/checkpoints/wav2lip.pth"
# Best val F1 per modality from 02_train_encoders_3emotions.ipynb (OUT_DIR there).
BEST_AUDIO_PATH = "/content/trained_encoders_3emotions/3emo-hubert-er-lr3e5"
BEST_VIDEO_PATH = "/content/trained_encoders_3emotions/3emo-tsf-lr3e5-16f-nf"
OUT_DIR = Path("/content/wav2lip_finetuned")
OUT_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE = {0, 1, 3, 5, 7}
REMAP = {2: 0, 4: 1, 6: 2}
EMOTIONS = ["happy", "angry", "disgust"]
# Used only when video encoder head has >3 classes (e.g. 6-class CREMA); 3-emo checkpoints use logits 0..2 directly.
WAV2LIP_TO_ENCODER = [1, 3, 5]

print(f"Device: {DEVICE}")

Device: cuda


In [20]:
IMG_SIZE = 96
MEL_STEP = 16
SR = 16000
FPS = 25

def wav_to_mel(wav_path, sr=SR):
    y, _ = librosa.load(wav_path, sr=sr)
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_mels=80, hop_length=200, win_length=800,
        fmin=55, fmax=7600)
    return librosa.power_to_db(mel, ref=np.max).astype(np.float32)


class Wav2LipDataset(Dataset):
    def __init__(self, metadata_path, split, T=5):
        with open(metadata_path) as f:
            data = json.load(f)
        self.samples = [s for s in data
                        if s["split"] == split and s["emotion_idx"] not in EXCLUDE]
        self.T = T

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]

        wav, sr = torchaudio.load(s["audio_path"])
        audio_1d = wav.squeeze(0)

        mel = wav_to_mel(s["audio_path"])

        frames = np.load(s["frames_path"]).astype(np.float32) / 255.0
        n_frames = frames.shape[0]

        start = np.random.randint(0, max(1, n_frames - self.T))
        face_window = frames[start:start + self.T]
        if face_window.shape[0] < self.T:
            pad = np.repeat(face_window[-1:], self.T - face_window.shape[0], axis=0)
            face_window = np.concatenate([face_window, pad], axis=0)

        mel_start = int(start / FPS * SR / 200)
        mel_end = mel_start + MEL_STEP * self.T
        mel_window = mel[:, mel_start:mel_end]
        if mel_window.shape[1] < MEL_STEP * self.T:
            mel_window = np.pad(mel_window, ((0, 0), (0, MEL_STEP * self.T - mel_window.shape[1])))

        gt = torch.from_numpy(face_window).permute(0, 3, 1, 2)
        H, W = gt.shape[2], gt.shape[3]
        if H != IMG_SIZE or W != IMG_SIZE:
            gt = F.interpolate(gt, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

        masked = gt.clone()
        masked[:, :, IMG_SIZE // 2:, :] = 0.0

        ref_idx = np.random.randint(0, n_frames)
        ref = torch.from_numpy(frames[ref_idx]).permute(2, 0, 1).unsqueeze(0).expand(self.T, -1, -1, -1)
        if ref.shape[2] != IMG_SIZE or ref.shape[3] != IMG_SIZE:
            ref = F.interpolate(ref, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)

        face_input = torch.cat([ref, masked], dim=1)

        mel_chunks = []
        for t in range(self.T):
            m = mel_window[:, t * MEL_STEP:(t + 1) * MEL_STEP]
            mel_chunks.append(torch.from_numpy(m).unsqueeze(0))
        mel_tensor = torch.stack(mel_chunks, dim=0)

        return {
            "mel": mel_tensor,
            "face_input": face_input,
            "gt": gt,
            "audio": audio_1d,
            "emotion": REMAP[s["emotion_idx"]],
        }


def collate_wav2lip(batch):
    return {
        "mel": torch.stack([b["mel"] for b in batch]),
        "face_input": torch.stack([b["face_input"] for b in batch]),
        "gt": torch.stack([b["gt"] for b in batch]),
        "audio": [b["audio"] for b in batch],
        "emotion": torch.tensor([b["emotion"] for b in batch]),
    }

In [21]:
def load_wav2lip(ckpt_path, device, freeze_encoders=True):
    model = Wav2LipModel()
    try:
        ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    except TypeError:
        ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt["state_dict"] if "state_dict" in ckpt else ckpt
    state = {k.replace("module.", ""): v for k, v in state.items()}
    model.load_state_dict(state, strict=False)
    if freeze_encoders:
        for p in model.face_encoder_blocks.parameters():
            p.requires_grad = False
        for p in model.audio_encoder.parameters():
            p.requires_grad = False
    return model.to(device)

wav2lip = load_wav2lip(WAV2LIP_CKPT, DEVICE)
total_params = sum(p.numel() for p in wav2lip.parameters())
trainable_params = sum(p.numel() for p in wav2lip.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params
print(f"Wav2Lip: {total_params/1e6:.1f}M total, {trainable_params/1e6:.1f}M trainable, {frozen_params/1e6:.1f}M frozen")

audio_enc, audio_proc = load_frozen_audio_encoder(BEST_AUDIO_PATH, DEVICE)
video_enc = load_frozen_video_encoder(BEST_VIDEO_PATH, DEVICE)
video_preprocess = DifferentiableVideoPreprocess(224).to(DEVICE)

VIDEO_ENC_FRAMES = getattr(video_enc.config, "num_frames", 8)
AUDIO_DIM = audio_enc.config.hidden_size
VIDEO_DIM = video_enc.config.hidden_size
PROJ_DIM = 256
emo_loss_fn = CrossModalEmotionLoss(weight=1.0)
print(f"Frozen encoders loaded. Video: {VIDEO_ENC_FRAMES} frames. "
      f"Audio dim={AUDIO_DIM}, Video dim={VIDEO_DIM}, Proj dim={PROJ_DIM}")
print("Emotion term: cross-modal cosine loss on projected embeddings (see diagram).")

AcceleratorError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
wandb.login()

CONFIGS = [
    {"name": "wav2lip-baseline", "weight_emo": 0.0},
    {"name": "wav2lip-emo-001",  "weight_emo": 0.01},
    {"name": "wav2lip-emo-002",  "weight_emo": 0.02},
    {"name": "wav2lip-emo-003",  "weight_emo": 0.03},
    {"name": "wav2lip-emo-004",  "weight_emo": 0.04},
    {"name": "wav2lip-emo-005",  "weight_emo": 0.05},
    {"name": "wav2lip-emo-01",   "weight_emo": 0.1},
]

LR = 1e-4
EPOCHS = 70
BATCH_SIZE = 16
PATIENCE = 5
T_FRAMES = 5
NUM_WORKERS = 2

In [ ]:
"""Fine-tuning loss (Wav2Lip face/audio encoders frozen; decoder trainable).\n
\n
L = mean_t L1(gen_t, gt_t) + weight_emo * mean_i (1 - cosine_sim(a_proj_i, v_proj_i))\n
\n
- Reconstruction: mean L1 between predicted and GT face crops per timestep, averaged over T frames.\n
- Emotion term: 256-d linear projections of frozen HuBERT (audio) vs frozen TimeSformer (generated video);\n
  mean 1 - cosine similarity. HuBERT runs under no_grad; gradients flow through Wav2Lip, video branch, both projections.\n
- Early stopping minimizes validation total (same L); test_loader is only for final held-out metrics after training.\n
"""

def adapt_frames(frames, target_t):
    """Resample (B, T, C, H, W) to (B, target_t, C, H, W) via uniform index sampling."""
    B, T, C, H, W = frames.shape
    if T == target_t:
        return frames
    idx = torch.linspace(0, T - 1, target_t, device=frames.device).long()
    return frames[:, idx]


def classify_gen_video(gen_frames, batch_emotions):
    """Optional monitoring: frozen TimeSformer classifier logits (not the training loss)."""
    gen_video = adapt_frames(gen_frames, VIDEO_ENC_FRAMES)
    pv = video_preprocess(gen_video)
    logits = video_enc(pixel_values=pv).logits
    n_lab = int(getattr(video_enc.config, "num_labels", logits.shape[-1]))
    if n_lab == len(EMOTIONS):
        logits_3 = logits
    else:
        logits_3 = logits[:, WAV2LIP_TO_ENCODER]
    labels_3 = batch_emotions.to(DEVICE)
    return logits_3, labels_3


def cross_modal_emo_loss(gen_video, batch_audio, audio_proj, video_proj):
    """Cosine loss: weight * mean(1 - cos(a, v)) on Linear(D_audio->256), Linear(D_video->256).
    Audio encoder @ no_grad; gradients flow through video branch + projections."""
    gen_adapt = adapt_frames(gen_video, VIDEO_ENC_FRAMES)
    with torch.no_grad():
        a_raw = extract_audio_embedding(audio_enc, audio_proc, batch_audio, device=DEVICE)
    a_p = audio_proj(a_raw)
    v_raw = extract_video_embedding(video_enc, video_preprocess, gen_adapt, device=DEVICE)
    v_p = video_proj(v_raw)
    return emo_loss_fn(a_p, v_p)


def train_one_epoch(model, loader, optimizer, scaler, weight_emo, audio_proj, video_proj):
    model.train()
    audio_proj.train()
    video_proj.train()
    total_recon, total_emo, total_loss = 0.0, 0.0, 0.0

    for batch in tqdm(loader, leave=False):
        mel = batch["mel"].to(DEVICE)
        face_in = batch["face_input"].to(DEVICE)
        gt = batch["gt"].to(DEVICE)
        B, T = mel.shape[0], mel.shape[1]

        optimizer.zero_grad(set_to_none=True)

        all_gen = []
        recon = 0.0
        with autocast("cuda", enabled=DEVICE == "cuda"):
            for t in range(T):
                gen = model(mel[:, t], face_in[:, t])
                recon += F.l1_loss(gen, gt[:, t])
                all_gen.append(gen)
            recon = recon / T

            emo = torch.tensor(0.0, device=DEVICE)
            if weight_emo > 0:
                gen_video = torch.stack(all_gen, dim=1)
                emo = cross_modal_emo_loss(gen_video, batch["audio"], audio_proj, video_proj)

            loss = recon + weight_emo * emo

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        params = list(filter(lambda p: p.requires_grad, model.parameters())) + list(audio_proj.parameters()) + list(video_proj.parameters())
        nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_recon += recon.item()
        total_emo += emo.item()
        total_loss += loss.item()

    n = len(loader)
    return {"recon": total_recon / n, "emotion": total_emo / n, "total": total_loss / n}


@torch.no_grad()
def evaluate(model, loader, weight_emo, audio_proj, video_proj):
    """Recon + cross-modal cosine emotion term; optional classifier accuracy for monitoring."""
    model.eval()
    audio_proj.eval()
    video_proj.eval()
    total_recon, total_emo, total_loss = 0.0, 0.0, 0.0
    correct, total_samples = 0, 0

    for batch in tqdm(loader, leave=False):
        mel = batch["mel"].to(DEVICE)
        face_in = batch["face_input"].to(DEVICE)
        gt = batch["gt"].to(DEVICE)
        B, T = mel.shape[0], mel.shape[1]

        all_gen = []
        recon = 0.0
        with autocast("cuda", enabled=DEVICE == "cuda"):
            for t in range(T):
                gen = model(mel[:, t], face_in[:, t])
                recon += F.l1_loss(gen, gt[:, t])
                all_gen.append(gen)
            recon = recon / T

            gen_video = torch.stack(all_gen, dim=1)

            emo = (
                cross_modal_emo_loss(gen_video, batch["audio"], audio_proj, video_proj)
                if weight_emo > 0
                else torch.tensor(0.0, device=DEVICE)
            )
            loss = recon + weight_emo * emo

            logits, enc_labels = classify_gen_video(gen_video, batch["emotion"])
            preds = logits.argmax(dim=1)
            correct += (preds == enc_labels).sum().item()
            total_samples += enc_labels.shape[0]

        total_recon += recon.item()
        total_emo += emo.item()
        total_loss += loss.item()

    n = len(loader)
    return {
        "recon": total_recon / n,
        "emotion": total_emo / n,
        "total": total_loss / n,
        "emo_accuracy": correct / total_samples if total_samples > 0 else 0,
        "mean_cosine_sim": 1.0 - (total_emo / n) if weight_emo > 0 and n > 0 else 0.0,
    }

In [ ]:
train_ds = Wav2LipDataset(METADATA, "train", T=T_FRAMES)
val_ds = Wav2LipDataset(METADATA, "val", T=T_FRAMES)
test_ds = Wav2LipDataset(METADATA, "test", T=T_FRAMES)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True,
                          collate_fn=collate_wav2lip)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True,
                        collate_fn=collate_wav2lip)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True,
                         collate_fn=collate_wav2lip)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

all_results = []

for cfg in CONFIGS:
    name = cfg["name"]
    weight_emo = cfg["weight_emo"]
    print(f"\n{'='*60}\n{name} (weight_emo={weight_emo})\n{'='*60}")

    wandb.init(project="uncanny-valley-wav2lip", name=name,
               config={**cfg, "lr": LR, "epochs": EPOCHS}, reinit=True)

    model = load_wav2lip(WAV2LIP_CKPT, DEVICE, freeze_encoders=True)
    audio_proj = nn.Linear(AUDIO_DIM, PROJ_DIM).to(DEVICE)
    video_proj = nn.Linear(VIDEO_DIM, PROJ_DIM).to(DEVICE)
    opt_params = list(filter(lambda p: p.requires_grad, model.parameters())) + list(audio_proj.parameters()) + list(video_proj.parameters())
    optimizer = torch.optim.AdamW(opt_params, lr=LR)
    scaler = GradScaler(enabled=DEVICE == "cuda")

    best_val, patience_cnt = float("inf"), 0
    save_path = OUT_DIR / name

    for epoch in range(EPOCHS):
        t = train_one_epoch(model, train_loader, optimizer, scaler, weight_emo, audio_proj, video_proj)
        v = evaluate(model, val_loader, weight_emo, audio_proj, video_proj)

        wandb.log({
            "epoch": epoch + 1,
            "train/recon": t["recon"], "train/emotion": t["emotion"], "train/total": t["total"],
            "val/recon": v["recon"], "val/emotion": v["emotion"], "val/total": v["total"],
            "val/emo_accuracy": v["emo_accuracy"],
            "val/mean_cosine_sim": v["mean_cosine_sim"],
        })

        print(f"  [{epoch+1:2d}/{EPOCHS}] "
              f"t_loss={t['total']:.4f} v_loss={v['total']:.4f} v_recon={v['recon']:.4f}"
              f" cos_sim={v['mean_cosine_sim']:.3f} cls_acc={v['emo_accuracy']:.3f}")

        if v["total"] < best_val:
            best_val = v["total"]
            save_path.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), save_path / "wav2lip.pth")
            torch.save(audio_proj.state_dict(), save_path / "audio_proj.pth")
            torch.save(video_proj.state_dict(), save_path / "video_proj.pth")
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    wandb.finish()
    del model, optimizer, scaler, audio_proj, video_proj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    all_results.append({"name": name, "weight_emo": weight_emo, "best_val": best_val})
    print(f"  Best val loss: {best_val:.4f} -> {save_path}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(all_results).sort_values("best_val")
print(df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df["name"], df["best_val"], color="steelblue")
ax.set_ylabel("Best Val Loss")
ax.set_title("Wav2Lip Fine-tuning: weight_emo ablation")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
best_name = df.iloc[0]["name"]
best_weight_emo = float(df.iloc[0]["weight_emo"])

def _load_state_dict(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

best_model = load_wav2lip(WAV2LIP_CKPT, DEVICE)
best_model.load_state_dict(_load_state_dict(OUT_DIR / best_name / "wav2lip.pth"))
best_model.eval()
audio_proj = nn.Linear(AUDIO_DIM, PROJ_DIM).to(DEVICE)
video_proj = nn.Linear(VIDEO_DIM, PROJ_DIM).to(DEVICE)
ap_path = OUT_DIR / best_name / "audio_proj.pth"
vp_path = OUT_DIR / best_name / "video_proj.pth"
if ap_path.is_file() and vp_path.is_file():
    audio_proj.load_state_dict(_load_state_dict(ap_path))
    video_proj.load_state_dict(_load_state_dict(vp_path))
else:
    print("Warning: projection checkpoints missing; emotion metrics may be wrong.")
print(f"Loaded best model: {best_name} (weight_emo={best_weight_emo})")

v_metrics = evaluate(best_model, val_loader, best_weight_emo, audio_proj, video_proj)
te_metrics = evaluate(best_model, test_loader, best_weight_emo, audio_proj, video_proj)
print(f"\nBest model — validation: recon={v_metrics['recon']:.4f}, emotion={v_metrics['emotion']:.4f}, total={v_metrics['total']:.4f}, "
      f"mean_cos_sim={v_metrics['mean_cosine_sim']:.4f}, cls_acc={v_metrics['emo_accuracy']:.4f}")
print(f"Best model — test (held-out): recon={te_metrics['recon']:.4f}, emotion={te_metrics['emotion']:.4f}, total={te_metrics['total']:.4f}, "
      f"mean_cos_sim={te_metrics['mean_cosine_sim']:.4f}, cls_acc={te_metrics['emo_accuracy']:.4f}")

del best_model, audio_proj, video_proj
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
"""Baseline vs Best: comparison with statistical significance (p-values)"""
from scipy import stats

baseline_name = df.loc[df["weight_emo"] == 0.0, "name"].iloc[0]
best_emo_df = df.loc[df["weight_emo"] > 0.0]
best_name = best_emo_df.iloc[0]["name"]
print(f"Baseline: {baseline_name}  |  Best emotion-aware: {best_name}")

def eval_model_per_sample(model, loader):
    """Per-sample L1 and per-sample correctness for statistical testing."""
    model.eval()
    sample_recons = []
    sample_correct = []
    correct_by_emo = {e: 0 for e in range(len(EMOTIONS))}
    total_by_emo = {e: 0 for e in range(len(EMOTIONS))}
    with torch.no_grad():
        for batch in tqdm(loader, leave=False, desc="Eval"):
            mel = batch["mel"].to(DEVICE)
            face_in = batch["face_input"].to(DEVICE)
            gt = batch["gt"].to(DEVICE)
            B, T = mel.shape[0], mel.shape[1]
            all_gen = []
            per_sample_recon = torch.zeros(B, device=DEVICE)
            for t in range(T):
                gen = model(mel[:, t], face_in[:, t])
                all_gen.append(gen)
                per_sample_recon += F.l1_loss(gen, gt[:, t], reduction="none").mean(dim=(1, 2, 3))
            per_sample_recon /= T
            sample_recons.extend(per_sample_recon.cpu().tolist())

            gen_video = torch.stack(all_gen, dim=1)
            logits, enc_labels = classify_gen_video(gen_video, batch["emotion"])
            preds = logits.argmax(dim=1)
            hits = (preds == enc_labels).cpu().tolist()
            sample_correct.extend(hits)
            for i, e in enumerate(batch["emotion"].tolist()):
                total_by_emo[e] += 1
                if hits[i]:
                    correct_by_emo[e] += 1
    total_correct = sum(correct_by_emo.values())
    total_samples = sum(total_by_emo.values())
    return {
        "recon": np.mean(sample_recons),
        "recon_samples": np.array(sample_recons),
        "emo_accuracy": total_correct / total_samples if total_samples > 0 else 0,
        "correct": np.array(sample_correct, dtype=bool),
        "by_emotion": {
            e: correct_by_emo[e] / total_by_emo[e] if total_by_emo[e] > 0 else 0
            for e in range(len(EMOTIONS))
        },
    }


def _load_state_dict(path):
    try:
        return torch.load(path, map_location=DEVICE, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=DEVICE)

baseline = load_wav2lip(WAV2LIP_CKPT, DEVICE)
baseline.load_state_dict(_load_state_dict(OUT_DIR / baseline_name / "wav2lip.pth"))

best = load_wav2lip(WAV2LIP_CKPT, DEVICE)
best.load_state_dict(_load_state_dict(OUT_DIR / best_name / "wav2lip.pth"))

print("Evaluating baseline (L1 only)...")
baseline_metrics = eval_model_per_sample(baseline, val_loader)
print("Evaluating best (L1 + emotion loss)...")
best_metrics = eval_model_per_sample(best, val_loader)

# --- Statistical significance ---
# L1 reconstruction: paired t-test & Wilcoxon signed-rank
_br = baseline_metrics["recon_samples"]
_bst = best_metrics["recon_samples"]
_n = min(len(_br), len(_bst))
_br, _bst = _br[:_n], _bst[:_n]
if _n < 2:
    t_stat, p_ttest = float("nan"), float("nan")
    w_stat, p_wilcox = float("nan"), float("nan")
else:
    t_stat, p_ttest = stats.ttest_rel(_br, _bst)
    try:
        w_stat, p_wilcox = stats.wilcoxon(_br, _bst)
    except ValueError:
        w_stat, p_wilcox = float("nan"), float("nan")

# Emotion accuracy: McNemar's test on paired correct/incorrect (same prefix length as L1)
b_ok = baseline_metrics["correct"][:_n]
e_ok = best_metrics["correct"][:_n]
n01 = int((b_ok & ~e_ok).sum())  # baseline correct, best wrong
n10 = int((~b_ok & e_ok).sum())  # baseline wrong, best correct
mcnemar_chi2 = (abs(n01 - n10) - 1) ** 2 / max(n01 + n10, 1)
p_mcnemar = 1 - stats.chi2.cdf(mcnemar_chi2, df=1) if (n01 + n10) > 0 else 1.0

print("\n=== Statistical significance ===")
print(f"L1 recon  — paired t-test: t={t_stat:.4f}, p={p_ttest:.4e}")
print(f"L1 recon  — Wilcoxon signed-rank: W={w_stat:.1f}, p={p_wilcox:.4e}")
print(f"Emo acc   — McNemar's test: χ²={mcnemar_chi2:.4f}, p={p_mcnemar:.4e}"
      f"  (n01={n01}, n10={n10})")

# --- Summary table ---
cmp = pd.DataFrame({
    "metric": ["L1 recon", "emo_accuracy"],
    baseline_name: [baseline_metrics["recon"], baseline_metrics["emo_accuracy"]],
    best_name: [best_metrics["recon"], best_metrics["emo_accuracy"]],
    "p-value": [p_wilcox, p_mcnemar],
})
cmp["delta"] = cmp[best_name] - cmp[baseline_name]
print("\n=== Baseline vs Best comparison ===")
print(cmp.to_string(index=False))

per_emo = pd.DataFrame({
    "emotion": EMOTIONS,
    f"{baseline_name}_acc": [baseline_metrics["by_emotion"][e] for e in range(len(EMOTIONS))],
    f"{best_name}_acc": [best_metrics["by_emotion"][e] for e in range(len(EMOTIONS))],
})
per_emo["delta"] = per_emo[f"{best_name}_acc"] - per_emo[f"{baseline_name}_acc"]
print("\n=== Per-emotion classification accuracy ===")
print(per_emo.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
x_agg = np.arange(2)
w_agg = 0.35
axes[0].bar(x_agg - w_agg/2, [baseline_metrics["recon"], baseline_metrics["emo_accuracy"]], w_agg, label=baseline_name, color="gray", alpha=0.7)
axes[0].bar(x_agg + w_agg/2, [best_metrics["recon"], best_metrics["emo_accuracy"]], w_agg, label=best_name, color="steelblue", alpha=0.7)
axes[0].set_xticks(x_agg)
axes[0].set_xticklabels(["L1 recon", "emo accuracy"])
axes[0].set_ylabel("Value")
axes[0].legend()
axes[0].set_title("Aggregate metrics")
for i, (bl, bst, pv) in enumerate(zip(
    [baseline_metrics["recon"], baseline_metrics["emo_accuracy"]],
    [best_metrics["recon"], best_metrics["emo_accuracy"]],
    [p_wilcox, p_mcnemar],
)):
    star = "***" if pv < 0.001 else "**" if pv < 0.01 else "*" if pv < 0.05 else "n.s."
    y_max = max(bl, bst)
    axes[0].text(i, y_max + 0.01, star, ha="center", fontsize=10)

x = np.arange(len(EMOTIONS))
w = 0.35
axes[1].bar(x - w/2, per_emo[f"{baseline_name}_acc"], w, label=baseline_name, color="gray", alpha=0.7)
axes[1].bar(x + w/2, per_emo[f"{best_name}_acc"], w, label=best_name, color="steelblue", alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels(EMOTIONS)
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].set_title("Per-emotion classification accuracy")
plt.tight_layout()
plt.show()

print("\n=== Side-by-side sample frames (one per emotion) ===")
best.eval()
one_per_emotion = {}
for batch in val_loader:
    for i in range(batch["emotion"].shape[0]):
        e = batch["emotion"][i].item()
        if e not in one_per_emotion:
            one_per_emotion[e] = {}
            for k, v in batch.items():
                if torch.is_tensor(v):
                    one_per_emotion[e][k] = v[i]
                elif isinstance(v, list):
                    one_per_emotion[e][k] = v[i]
                else:
                    one_per_emotion[e][k] = v
    if len(one_per_emotion) == len(EMOTIONS):
        break

fig, axes = plt.subplots(len(EMOTIONS), 4, figsize=(10, 2.5 * len(EMOTIONS)))
for row, e in enumerate(range(len(EMOTIONS))):
    if e not in one_per_emotion:
        continue
    sample = one_per_emotion[e]
    mel = sample["mel"].unsqueeze(0).to(DEVICE)
    face_in = sample["face_input"].unsqueeze(0).to(DEVICE)
    gt = sample["gt"].unsqueeze(0).to(DEVICE)
    T = mel.shape[1]
    with torch.no_grad():
        base_gen = [baseline(mel[:, t], face_in[:, t]) for t in range(T)]
        best_gen = [best(mel[:, t], face_in[:, t]) for t in range(T)]
    mid = T // 2
    axes[row, 0].imshow(gt[0, mid].permute(1, 2, 0).cpu().clamp(0, 1))
    axes[row, 0].set_title(f"{EMOTIONS[e]} (GT)")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(base_gen[mid][0].permute(1, 2, 0).cpu().clamp(0, 1))
    axes[row, 1].set_title("baseline")
    axes[row, 1].axis("off")
    axes[row, 2].imshow(best_gen[mid][0].permute(1, 2, 0).cpu().clamp(0, 1))
    axes[row, 2].set_title("best (emo)")
    axes[row, 2].axis("off")
    diff = (best_gen[mid][0] - base_gen[mid][0]).abs().mean(dim=0).cpu()
    axes[row, 3].imshow(diff, cmap="hot")
    axes[row, 3].set_title("|diff|")
    axes[row, 3].axis("off")
plt.suptitle("Baseline vs emotion-aware: sample frame per emotion")
plt.tight_layout()
plt.show()

del baseline, best
if torch.cuda.is_available():
    torch.cuda.empty_cache()